In [72]:
from langchain_openai import ChatOpenAI


from dotenv import load_dotenv


load_dotenv()


from langchain.document_loaders import PyPDFDirectoryLoader


from langchain.text_splitter import RecursiveCharacterTextSplitter


# from langchain.embeddings.openai import OpenAIEmbeddings


from langchain_openai import OpenAIEmbeddings


from langchain_pinecone import PineconeVectorStore


In [73]:
import os


In [74]:
def read_doc(directory):
    file_loader = PyPDFDirectoryLoader(directory)
    documents = file_loader.load()
    return documents


In [75]:
docs = read_doc("../artifacts/")


In [76]:
len(docs)


62

In [77]:
## Divide the docs into chunks


def chunk_data(docs, chunk_size=800, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    doc = text_splitter.split_documents(docs)
    return doc


In [78]:
documents = chunk_data(docs)


In [79]:
print(type(documents), len(documents))


<class 'list'> 154


In [80]:
embeddings = OpenAIEmbeddings(
    openai_api_key=os.getenv("OPENAI_API_KEY"), model="text-embedding-3-small"
)
embeddings


OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000216EA0F34F0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000216EC16D3F0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [81]:
vectors = embeddings.embed_query("How are you?")


In [82]:
type(embeddings)


langchain_openai.embeddings.base.OpenAIEmbeddings

In [83]:
len(vectors)


1536

In [84]:
## Vector Search DB In Pinecone
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY"),
    environment="us-east-1",
)

index_name = "langchainvector"


In [85]:
# Connect to the Pinecone index using LangChainIs Pinecone wrapper
# Add all the split documents into the Pinecone vector database
vector_store = PineconeVectorStore(index_name=index_name, embedding=embeddings)


In [86]:
vector_store.add_documents(documents)


['ef20ffda-2cb1-4f5b-b001-ae1a78b708c1',
 '9caea0e9-7bf6-46fc-bfb6-ee34e6a93e66',
 '4f6df352-57a1-48c3-bdcd-9f9c0fa3463a',
 '157471c5-fc41-4b22-98eb-52078db6b55b',
 'dd583b7b-4f81-4333-99ce-7ad2784aa893',
 'a2ce6575-d5a9-4ae2-815c-829587179573',
 '1e037ed8-a793-43f5-afdb-2458e4e927b3',
 'ebd8852c-1dfb-45ca-9a60-ff290bd2d294',
 'b2e39db4-e3aa-4da5-a848-e740ee7ad277',
 '2ad50414-7777-4fe8-9d98-e8a0215ecf5a',
 'b1ecb1ab-555b-46f1-aa15-e18b63b3a758',
 'c2b3770d-7f57-47c6-ab68-ccd1bea8aef8',
 '8f052e0e-57c9-4b9e-86a6-0920f612861d',
 '20e93340-f2e8-4bb9-a05b-ab3d1dc4a152',
 '11024660-b812-4690-96ff-49fbe9faeb36',
 'c2cc7644-c303-48d4-b52d-7281fce9d9c0',
 '4943d23a-0cf2-4daa-987a-7482666fe2ef',
 '0a956e8d-ab02-442b-abb1-60045c48512f',
 '1cb4f713-6e61-413a-b4c8-7591b066d63e',
 '6903da96-ba8e-4c8d-84f6-96f498afeb19',
 'e19f9e8a-dae9-4825-bb34-18609ee4972d',
 '56bdf9df-0336-443f-a617-48829c4303b8',
 'e264ee16-64d2-41f0-82d3-6c15c3a7f2f1',
 '09f99755-5e5c-4fe3-8488-d5331b9ad6d2',
 '2ef16121-cded-

In [87]:
## Cosine Similarity Retreive Results from VectorDB


def retrieve_query(query, k=2):
    matching_results = vector_store.similarity_search(query, k=k)
    return matching_results


In [88]:
from langchain.chains.question_answering.chain import (
    load_qa_chain,
)  # for question answering


In [99]:
llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-3.5-turbo",
    temperature=0.5,
)
chain = load_qa_chain(
    llm, chain_type="stuff"
)  # To create a question answering application


In [100]:
## Search answers from VectorDB


def retrieve_answers(query):
    doc_search = retrieve_query(query)
    print(doc_search)
    response = chain.run(input_documents=doc_search, question=query)
    print(response)
    return response  # Ensure the correct key is used to extract the answer


In [101]:
our_query = "give me last year's financial report"
answer = retrieve_answers(our_query)


[Document(id='874fdd33-edb5-4ca8-97d4-0981af79807f', metadata={'page': 24.0, 'source': '..\\artifacts\\budget_speech.pdf'}, page_content='21  \n \n113. The gross and net market borrowings through dated securities during \n2024-25 are estimated at ` 14.01 lakh crore  and ` 11.63 lakh crore \nrespectively. Both will be less than that in 2023-24.  \n114. The fiscal consolidation path announced by me in 2021 has served our \neconomy very well, and we aim to reach a deficit below 4.5 per cent next \nyear. The Government is committed to staying the course. From 2026 -27 \nonwards, our endeavour will be to keep the fiscal deficit each year such that \nthe Central Government debt will be on a declining path as percentage of \nGDP.  \nI will, now, move to Part B.'), Document(id='631a051a-63de-4f4b-9218-35ef1d2e4ffb', metadata={'page': 24.0, 'source': '..\\artifacts\\budget_speech.pdf'}, page_content='21  \n \n113. The gross and net market borrowings through dated securities during \n2024-25 are

In [103]:
print(answer)


Based on the provided context, the financial report for the previous year (2023-24) states that the estimated gross and net market borrowings through dated securities were higher than the projected amounts for the upcoming year (2024-25). The fiscal deficit was targeted to be below 4.5 per cent for the next year, with a commitment to fiscal consolidation. The government aims to ensure that the Central Government debt as a percentage of GDP follows a declining path from 2026-27 onwards.
